In [31]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import skfuzzy as fuzz
from skimage.segmentation import find_boundaries
from scipy.spatial.distance import directed_hausdorff
import numpy as np

In [32]:
BASE = "GroundTruths_bmpformat"

IMG_PATH = "/content/drive/MyDrive/Test_bmp"
GT_PATH  = "/content/drive/MyDrive/Test_GT_bmp"

files = sorted(os.listdir(IMG_PATH))[:50]
files1 = sorted(os.listdir(GT_PATH))[:50] #Considering only the first 50 images

In [33]:
def load_pair(name):
    img = cv2.imread(os.path.join(IMG_PATH,name))
    gt  = cv2.imread(os.path.join(GT_PATH,name),0) #Reading ground truth as only grayscale, since we separate image based on given values anyway
    return img,gt

def split_gt(gt):
    #Splitting the image into nucleus and cytoplasm as given in the dataset
    nuc = (gt==100).astype(np.uint8)
    cyt = (gt==255).astype(np.uint8)
    return nuc,cyt

def crop_to_cell(img, n_gt, c_gt, pad=5):
    #Using this function to focus on the WBC part alone, rather than the surrounding RBCs
    coords = np.column_stack(np.where(c_gt > 0))
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    y_min = max(y_min - pad, 0)
    x_min = max(x_min - pad, 0)
    y_max = min(y_max + pad, img.shape[0])
    x_max = min(x_max + pad, img.shape[1])
    return (img[y_min:y_max, x_min:x_max], n_gt[y_min:y_max, x_min:x_max], c_gt[y_min:y_max, x_min:x_max])

In [34]:
def hausdorff(a, b):
    #Hausdroff distance for calculating boundary accuracy, measures maximum boundary deviation between predicted and ground-truth, capturing worst-case boundary error; a lower value indicates better boundary alignment.
    A = np.column_stack(np.nonzero(a))
    B = np.column_stack(np.nonzero(b))
    return max(directed_hausdorff(A, B)[0], directed_hausdorff(B, A)[0])

In [35]:
def kmeans_seg(img,k):
    X = img.reshape((-1,3)).astype(np.float32) #Converting all pixels to samples x features
    km = KMeans(n_clusters=k,n_init=10).fit(X)
    return km.labels_.reshape(img.shape[:2]) #Reshaping to images

def fcm_seg(img, k):
    X = img.reshape((-1,3)).T.astype(np.float32)
    centers, membership, _,_,_,_,_ = fuzz.cluster.cmeans(X, k, 2, error=0.005, maxiter=1000) #Maintaining m=2, the generally accepted value
    return centers, membership

In [36]:
def map_clusters(pred, gt_n, gt_c):
    nuc_score=[]
    cyt_score=[]
    cluster_ids = np.unique(pred)

    for c in cluster_ids:
        mask = pred == c
        nuc_score.append(np.sum(mask & gt_n))
        cyt_score.append(np.sum(mask & gt_c))

    n_id = cluster_ids[np.argmax(nuc_score)]
    c_id = cluster_ids[np.argmax(cyt_score)]

    return ((pred == n_id).astype(np.uint8), (pred == c_id).astype(np.uint8), n_id, c_id)

In [37]:
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7] #Having variable thresholds for acccepting as nucleus or cytoplasm
k_vals = [2, 3] #Different no. of clusters

results = []
count = 1

for k in k_vals:
    for f in files:
        img, gt = load_pair(f)
        n_gt, c_gt = split_gt(gt)
        img, n_gt, c_gt = crop_to_cell(img, n_gt, c_gt)

        km = kmeans_seg(img, k)
        km_n, km_c, _, _ = map_clusters(km, n_gt, c_gt)
        knd = hausdorff(n_gt, km_n)
        knc = hausdorff(c_gt, km_c)

        centers, membership = fcm_seg(img, k)
        labels_hard = np.argmax(membership, axis=0).reshape(img.shape[:2]) #Just for obtaining the cluster ids, after which we consider soft clustering using membership values
        _, _, n_id, c_id = map_clusters(labels_hard, n_gt, c_gt)

        nucleus_prob = membership[n_id].reshape(img.shape[:2])
        cyto_prob = membership[c_id].reshape(img.shape[:2])

        for t in thresholds:
            fcm_n = (nucleus_prob > t).astype(np.uint8)
            fcm_c = (cyto_prob > t).astype(np.uint8)
            results.append({"k": k, "threshold": t, "knd": knd, "kcd": knc, "fnd": hausdorff(n_gt, fcm_n), "fcd": hausdorff(c_gt, fcm_c)})

df = pd.DataFrame(results)
mean_hd = df.groupby(["k", "threshold"]).mean()
print("Mean Hausdorff Distance per k and threshold:")
print(mean_hd)

Mean Hausdorff Distance per k and threshold:
                   knd        kcd         fnd        fcd
k threshold                                             
2 0.3        98.013842  73.687830  106.455802  75.837549
  0.4        98.013842  73.687830  104.152646  75.161793
  0.5        98.013842  73.687830  100.750200  74.289600
  0.6        98.013842  73.687830   91.133766  73.387756
  0.7        98.013842  73.687830   80.063174  72.439348
3 0.3        36.765266  69.483158   64.633364  70.503540
  0.4        36.765266  69.483158   49.740162  69.478381
  0.5        36.765266  69.483158   37.220453  68.641127
  0.6        36.765266  69.483158   26.836376  68.185403
  0.7        36.765266  69.483158   19.895264  67.391590


From the results, soft clustering clearly provides better segmentation compared to hard clustering when properly tuned. With k = 2, the segmentation simply separates into nucleus and cytoplasm, while k = 3 introduces an additional cluster that captures boundary variations or background noise within the region of interest, which we have achieved by cropping to the cytoplasm of the WBC, so as to avoid the surrounding RBCs. This improved segmentation due to added background cluster also explains the significant drop in Hausdorff distance when moving from k = 2 to k = 3. As mentioned in the code, we use Hausdroff distance for calculating boundary accuracy, which measures maximum boundary deviation between predicted and ground-truth, capturing worst-case boundary error with a lower value indicates better boundary alignment, although it maybe prone to outliers. In the case of Fuzzy C-Means, increasing the membership threshold progressively removes uncertain boundary pixels, leading to tighter predicted boundaries and hence, lower Hausdorff values, while K Means scores' remains unaffected since it performs hard clustering, so thresholds do not matter. The lowest boundary error is achieved with k = 3 and a higher threshold of 0.7, indicating that soft clustering, when combined with an appropriate number of clusters and threshold value, yields better segmentation and boundary accuracy compared to hard clustering.